In [ ]:
import awswrangler as wr

import mlflow

# Para que funciones, todos nuestros scripts debemos exportar las siguientes variables de entorno
%env AWS_ACCESS_KEY_ID=minio   
%env AWS_SECRET_ACCESS_KEY=minio123 
%env MLFLOW_S3_ENDPOINT_URL=http://localhost:9000
%env AWS_ENDPOINT_URL_S3=http://localhost:9000
#%env MLFLOW_S3_ENDPOINT_URL=http://192.168.0.21:9000
#%env AWS_ENDPOINT_URL_S3=http://192.168.0.21:9000

In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from astropy.time import Time
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
import optuna
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
mlflow_server = "http://localhost:5001"
mlflow.set_tracking_uri(mlflow_server)

In [ ]:
data = wr.s3.read_csv("s3://data/data.csv")

In [ ]:
train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)
# Print the shapes of the resulting DataFrames
print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

# Now you can work with train_df and test_df for training and evaluation
X_train = train_df.drop(['class'], axis=1)
y_train = train_df['class']
X_test = test_df.drop(['class'], axis=1)
y_test = test_df['class']

In [ ]:
numerical_cols = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift", "Month"]
#categorical_cols = ["class", "cam_col"]

scaler = StandardScaler()

train_df_norm = scaler.fit_transform(X_train[numerical_cols])

test_df_norm = scaler.transform(X_test[numerical_cols])

In [ ]:
experiment_name = "KnnRandom"

if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(name=experiment_name) 

experiment = mlflow.get_experiment_by_name(experiment_name)

In [ ]:
neigh = KNeighborsClassifier(n_neighbors=3)
df_train = pd.DataFrame(train_df_norm, columns=numerical_cols)
df_test = pd.DataFrame(test_df_norm, columns=numerical_cols)
neigh.fit(df_train[numerical_cols], y_train)
y_pred = neigh.predict(df_test[numerical_cols])
print("Reporte de clasificación:")
print(classification_report(test_df["class"], y_pred))

print("Precisión del modelo:", accuracy_score(test_df["class"], y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Galaxy', 'QSO', 'Star'], yticklabels=['Galaxy', 'QSO', 'Star'])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
df_train = pd.DataFrame(train_df_norm, columns=numerical_cols)
df_test = pd.DataFrame(test_df_norm, columns=numerical_cols)

model = KNeighborsClassifier()
param_distributions = {
    'n_neighbors': np.arange(1, 30, 1),          # Number of neighbors to consider
    'weights': ['uniform', 'distance'],          # Weight function
    'metric': ['euclidean', 'manhattan', 'minkowski']  # Distance metrics
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=20,              # Number of random combinations to try
    scoring='accuracy',     # Optimization metric
    cv=5,                   # 5-fold cross-validation
    random_state=42,
    n_jobs=-1               # Use all available CPU cores
)

random_search.fit(df_train, y_train)

print("Best Hyperparameters:", random_search.best_params)
print("Best Cross-Validation Accuracy:", random_search.best_score)
best_model = random_search.best_estimator
y_pred = best_model.predict(df_test)
test_accuracy = accuracy_score(y_test, y_pred)
print("Test Set Accuracy:", test_accuracy)

In [ ]:
print("Reporte de clasificación:")
print(classification_report(test_df["class"], y_pred))

print("Precisión del modelo:", accuracy_score(test_df["class"], y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Galaxy', 'QSO', 'Star'], yticklabels=['Galaxy', 'QSO', 'Star'])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
print(f'Los mejores parámetros son: {random_search.best_params_}')

In [ ]:
from sklearn.metrics import precision_score, accuracy_score, recall_score
with mlflow.start_run(experiment_id = experiment.experiment_id):
    # Se registran los mejores hiperparámetros
    mlflow.log_params(random_search.best_params_)
    
    # Se obtiene las predicciones del dataset de evaluación
    y_pred = random_search.predict(X_test)
    
    # Se calculan las métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    
    # Y las enviamos a MLFlow
    metrics ={
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall 
        }
    mlflow.log_metrics(metrics)
    
    # Como artefactos, obtenemos las gráficas de la curva ROC y la matriz de confusion
    matrix_plot = plot_confusion_matrix(y_test, y_pred, save_path=None)
    roc_plots = plot_roc_curve(y_test, y_pred, save_path=None)
    
    mlflow.log_figure(matrix_plot, artifact_file="matrix_plot.png")
    mlflow.log_figure(roc_plots[0], artifact_file="roc_curve_1_plot.png")
    mlflow.log_figure(roc_plots[1], artifact_file="roc_curve_2_plot.png")
    mlflow.log_figure(roc_plots[2], artifact_file="roc_curve_3_plot.png")
    
    # Registramos el modelo y los datos de entrenamiento
    mlflow.sklearn.log_model(random_search, 'KnnRandom')